In [ ]:
import sys
import os
import logging

import torch
from transformers.image_utils import to_numpy_array
import numpy as np
import pdb

from experiments import Exp2
from dataset import Imdb_Dataloader
from config import MODEL_NAMES
from models.model_registry import get_model
# from utils import parse_args
import argparse
parser = argparse.ArgumentParser()

# ROOT = os.path.dirname(os.path.abspath(__file__))
# sys.path.insert(0, ROOT)
logger = logging.getLogger(__name__)

# add experiment args:
if not 'base_args' in locals():
    base_args = [x for x in sys.argv]

# with sobel_channelwise threshold=1.0, eps=0.2, works well at flat lr=0.03 but takes too long, doesn't quite succeed after 200 steps
# with sobel_channelwise unscaled threshold=0.05, at flat lr=0.03 seems to bounce around just above loss=0 after 200 steps


# cosine step_size 0.05 and 300 steps works well with eps=0.2, threshold=0.2, scales=1,2,3
# with eps 0.15, step_size = 0.05 doesn't cut it after 500 steps
# reducing threshold to 0.15 doesn't change that, but at 0.1 it happens after 150/500 cosine steps (still sobel)
# with robust_canny instead of robust_sobel, similar, but width=2 is too generous both for text and fuzzy logo
# with width=1, loss never reaches 0 after 500 cosine 0.05 steps
# unless eps is changed from 0.15 to 0.2 to compensate (getting there with 150/500 cosine steps)
# with width=3, threshold=0.1 and eps=0.15 is plenty

# cmd_args = [     '--dataset_path',  '/home/abarsky/data/advdoc_data_nsampl1000_nqst5.pkl', # base_cmd["dataset_path"],
#                  '--target_answer', ['smoke weed and hail satan'], 
#                  '--n_questions',   1,
#                  '--device',        'cuda:0',
#                  '--model',         'pix2struct',
#                  '--eps',           0.15, # 255, # 96,
#                  '--step_size',     0.05,   # (8*4)/255 ,   # 8  #(for ints)
#                  '--steps',         500,  
#                  '--seed',          42,
#                  '--mask',          'sobel_channelwise', # channelwise with scale_threshold needs threshold to be about 10-20x bigger
#                  '--mask_kwargs',   {'threshold': 0.1, 'scales': [1,2,3], 'width': 3},
#                  '--scheduler',     'cosine_annealing', # cosine_annealing or no_scheduler
#                  '--perceptual',    True, # use perceptual manipulation (applies eps in LAB space, step_size is in float rather than int units)
#                  '--tracking',      True, # show tensorboard trackers
#            ]


### with sobel threshold = 0.1 (moderate), we can get there after ~115/150 cosine steps with epsilon 0.3 (moderate)
### or 150/200 steps with epsilon 0.15 (low)
### with sobel threshold = 0.02 (low, many pixels) we can get there with epsilon 0.1 (v. low) after 100/150 steps
# cmd_args = [     '--dataset_path',  '/home/abarsky/data/advdoc_data_nsampl1000_nqst5.pkl', # base_cmd["dataset_path"],
#                  '--target_answer', ['smoke weed and hail satan'], 
#                  '--n_questions',   1,
#                  '--device',        'cuda:0',
#                  '--model',         'donut',
#                  '--eps',           0.15, # 255, # 96,
#                  '--step_size',     0.03,   # (8*4)/255 ,   # 8  #(for ints)
#                  '--steps',         200,  
#                  '--seed',          42,
#                  '--mask',          'sobel',
#                  '--mask_kwargs',   {'threshold': 0.1},
#                  '--scheduler',     'cosine_annealing', # cosine_annealing or no_scheduler
#                  '--perceptual',    True, # use perceptual manipulation (applies eps in LAB space, step_size is in (oklab)float rather than int units)
#                  '--tracking',      False, # show tensorboard trackers
#            ]

#### 114 steps with these args:
# cmd_args = [     '--dataset_path',  '/home/abarsky/data/advdoc_data_nsampl1000_nqst5.pkl', # base_cmd["dataset_path"],
#                  '--target_answer', ['smoke weed and hail satan'], 
#                  '--n_questions',   1,
#                  '--device',        'cuda:0',
#                  '--model',         'donut',
#                  '--eps',           0.3, # 255, # 96,
#                  '--step_size',     0.01,   # (8*4)/255 ,   # 8  #(for ints)
#                  '--steps',         300,   # 0.05 gets stuck at loss=6. after 100/200 cosine steps
#                  '--seed',          42,
#                  '--mask',          'sobel',
#                  '--mask_kwargs',   {'threshold': 0.2},
#                  '--scheduler',     'cosine_annealing', # cosine_annealing or no_scheduler
#                  '--perceptual',    True, # use perceptual manipulation (applies eps in LAB space, step_size is in float rather than int units)
#                  '--tracking',      True, # show tensorboard trackers
#            ]

#### non perceptual baseline:
cmd_args = [     '--dataset_path',  '/home/abarsky/data/advdoc_data_nsampl1000_nqst5.pkl', # base_cmd["dataset_path"],
                 '--target_answer', ['smoke weed and hail satan'], 
                 '--n_questions',   1,
                 '--device',        'cuda:0',
                 '--model',         'pix2struct',
                 '--eps',           96,
                 '--step_size',     2,
                 '--steps',         300,   # 0.05 gets stuck at loss=6. after 100/200 cosine steps
                 '--seed',          42,
                 '--mask',          'sobel',
                 '--mask_kwargs',   {},
                 '--scheduler',     'no_scheduler', # cosine_annealing or no_scheduler
                 '--perceptual',    False, # use perceptual manipulation (applies eps in LAB space, step_size is in float rather than int units)
                 '--tracking',      False, # show tensorboard trackers
           ]



# if __name__ == '__main__':
if True:
    # args = parser.parse_args(cmd_args)
    args = argparse.Namespace(**{
        cmd_args[i].lstrip('-'): cmd_args[i+1]
        for i in range(0, len(cmd_args), 2)
    })
    # for arg, val in vars(args).items():
    #     print(f'{arg}: {val}')

    
    data_loader = Imdb_Dataloader(args.dataset_path).load_data()

    for kwarg, val in args._get_kwargs():
        print(f"{kwarg}: {val}")

    logger.info(f'Starting Experiment n.2')

    processor, autoprocessor, model, attack, mask_function = get_model(args.model, args)
    exp = Exp2(attack=attack, 
               model=model, 
               autoprocessor=autoprocessor, 
               processor=processor, 
               data_loader=data_loader,
               n_questions=args.n_questions,
               mask_function=mask_function,
               args=args)
    exp.setup()